# CIFAR100 Parallel Experiments - SFL, SFL Gold, Centinel

In [ ]:
import os
import subprocess
import concurrent.futures
from itertools import product
import json
import matplotlib.pyplot as plt
import glob
from matplotlib.ticker import MaxNLocator

os.makedirs('results', exist_ok=True)

### Generate Experiment Configurations

In [ ]:
baselines = ['sfl', 'sfl_gold', 'centinel']
malicious_fractions = [0.0, 0.7]
attack_types = ['backdoor', 'pair_flip']
partitions = ['iid', 'non_iid']

configs = []
for b, m, a, p in product(baselines, malicious_fractions, attack_types, partitions):
    # If malicious fraction is 0.0, the attack type doesn't matter, we don't need to run it multiple times.
    if m == 0.0:
        if a != 'backdoor': # Only keep one 'none' equivalent baseline per partition
            continue
        effective_attack = 'none'
    else:
        effective_attack = a
        
    configs.append({
        "baseline": b,
        "malicious_fraction": m,
        "attack_type": effective_attack,
        "partition": p
    })

# Deduplicate configs
unique_configs = []
seen = set()
for c in configs:
    identifier = f"{c['baseline']}_{c['malicious_fraction']}_{c['attack_type']}_{c['partition']}"
    if identifier not in seen:
        seen.add(identifier)
        unique_configs.append(c)

print(f"Total unique experiments to run: {len(unique_configs)}")
for i, conf in enumerate(unique_configs):
    print(f"Experiment {i}: {conf}")

### Run Experiments in Parallel (2 GPUs)

In [ ]:
def run_experiment(gpu_id, config, rounds=50):
    fname = f"results/{config['baseline']}_{config['malicious_fraction']}_{config['attack_type']}_{config['partition']}.json"
    if os.path.exists(fname):
        print(f"Skipping {fname}, already exists.")
        return fname
        
    cmd = [
        "python", "experiment_worker.py",
        "--baseline", config["baseline"],
        "--malicious_fraction", str(config["malicious_fraction"]),
        "--attack_type", config["attack_type"],
        "--partition", config["partition"],
        "--gpu", str(gpu_id),
        "--output_file", fname,
        "--rounds", str(rounds)
    ]
    
    print(f"Spawning on GPU {gpu_id}: {' '.join(cmd)}")
    # Note: subprocess.run blocks the thread until done
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"Error in experiment {fname}:\n{res.stderr}")
    return fname

MAX_WORKERS = 2 # 2x T4 GPUs

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = []
    for i, config in enumerate(unique_configs):
        gpu_id = i % 2
        futures.append(executor.submit(run_experiment, gpu_id, config, 100)) # Change rounds here as needed
        
    for future in concurrent.futures.as_completed(futures):
        print(f"Completed: {future.result()}")
print("All experiments finished!")

### Visualize Results

In [ ]:
def plot_comparison(partition, attack, m_frac=0.7):
    files = glob.glob(f"results/*_{m_frac}_{attack}_{partition}.json")
    files += glob.glob(f"results/*_0.0_none_{partition}.json")

    results = {}
    for f in files:
        with open(f, 'r') as fp:
            data = json.load(fp)
            conf = data['config']
            name = f"{conf['baseline']} (M={conf['malicious_fraction']})"
            results[name] = data

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    for name, data in results.items():
        rounds = range(1, len(data['test_acc']) + 1)
        ax1.plot(rounds, data['test_acc'], label=name)
        ax2.plot(rounds, data['asr'], label=name)

    ax1.set_title(f"Test Accuracy - {attack} - {partition}")
    ax1.set_xlabel("Round")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(True)

    ax2.set_title(f"Attack Success Rate - {attack} - {partition}")
    ax2.set_xlabel("Round")
    ax2.set_ylabel("ASR")
    ax2.legend()
    ax2.grid(True)

    plt.show()

plot_comparison('iid', 'backdoor')
plot_comparison('non_iid', 'backdoor')
plot_comparison('iid', 'pair_flip')
plot_comparison('non_iid', 'pair_flip')


### Results Table (Last 3 Rounds)
Mean and 95% Confidence Interval for Train Acc, Test Acc, and ASR.

In [ ]:
import glob
import json
import pandas as pd
import numpy as np
import scipy.stats

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    if n == 0:
        return 0.0, 0.0
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n-1) if n > 1 else 0.0
    return m * 100, h * 100

def get_stats_string(data):
    if len(data) == 0:
        return "N/A"
    m, h = mean_confidence_interval(data)
    return f"{m:.2f} ± {h:.2f}%"

def print_results_table():
    files = glob.glob("results/*.json")
    if not files:
        print("No results found.")
        return

    table_data = []
    
    for f in files:
        with open(f, 'r') as fp:
            data = json.load(fp)
            conf = data['config']
            
            train_acc_last3 = data['train_acc'][-3:] if len(data['train_acc']) >= 3 else data['train_acc']
            test_acc_last3 = data['test_acc'][-3:] if len(data['test_acc']) >= 3 else data['test_acc']
            asr_last3 = data['asr'][-3:] if len(data['asr']) >= 3 else data['asr']
            
            row = {
                "Baseline": conf['baseline'],
                "Malicious Fraction": conf['malicious_fraction'],
                "Attack Type": conf['attack_type'],
                "Partition": conf['partition'],
                "Train Acc": get_stats_string(train_acc_last3),
                "Test Acc": get_stats_string(test_acc_last3),
                "ASR": get_stats_string(asr_last3)
            }
            table_data.append(row)
            
    df = pd.DataFrame(table_data)
    df = df.sort_values(by=["Partition", "Attack Type", "Malicious Fraction", "Baseline"])
    display(df)

print_results_table()
